In [0]:
from pyspark.sql.functions import current_timestamp,col

SCHEMA_PATH = 'abfss://fraudmlops@fraudmlopsstorage.dfs.core.windows.net/schema/transactions_bronze'

SOURCE_PATH = 'abfss://fraudmlops@fraudmlopsstorage.dfs.core.windows.net/landing/transactions'

CHECKPOINT_PATH = 'abfss://fraudmlops@fraudmlopsstorage.dfs.core.windows.net/checkpoints/transactions_bronze'

BRONZE_TABLE = 'fraud_mlops.bronze.fraud_transactions_incremental'

raw_stream = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
        .option('cloudFiles.schemaLocation',SCHEMA_PATH)\
            .load(SOURCE_PATH)

bronze_stream = raw_stream.withColumn('ingested_at',current_timestamp())\
    .withColumn('source_file',col('_metadata.file_path'))

query = bronze_stream.writeStream.format('delta').outputMode('append')\
    .option('checkpointLocation',CHECKPOINT_PATH)\
        .trigger(availableNow=True)\
            .toTable(BRONZE_TABLE)



query.awaitTermination()

print("ADLS -> Bronze incremental ingestion completed.")




In [0]:
%sql
SELECT COUNT(*) FROM fraud_mlops.bronze.fraud_transactions_incremental
